# Movie Recommendation System

This notebook has two parts:

1. **Content-based recommender** (the main task): uses the TMDB 5000 dataset to recommend
   movies similar to a given one. It produces `model.pkl` and `similarity.pkl`, which `app.py` uses.
2. **Bonus**: user-based collaborative filtering and a **hybrid** recommender, built on the
   MovieLens dataset (ml-latest-small) and evaluated with metrics.

The TMDB CSV files must be in `data/`. MovieLens is downloaded automatically the first time.

In [ ]:
import ast
import os
import pickle
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import truststore
truststore.inject_into_ssl()   # use Windows' certificate store (fixes SSL errors behind antivirus/proxies)

DATA_DIR = 'data'

# Content model options.
# VECTORIZER can be 'tfidf' or 'count'. Both are compared in the evaluation section.
VECTORIZER = 'tfidf'
MAX_FEATURES = 5000   # vocabulary size (previously 500, which is too small)
N_RECS = 10
RANDOM_STATE = 42

## Part 1 · Content-based recommender

### Step 1: Load and explore the data

In [ ]:
movies_raw = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_movies.csv'))
credits = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_credits.csv'))
print('movies :', movies_raw.shape)
print('credits:', credits.shape)
movies_raw.head(2)

movies : (4803, 20)
credits: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


**Change from the original version:** the two tables used to be merged on `title`.
Some titles are shared by different movies (e.g. *Batman* from 1966 and from 1989), so
merging on title cross-matched their rows and created fake combinations (one movie's cast
with the other movie's overview). The tables are now merged on the **TMDB ID**, which is
unique: it is called `id` in `movies` and `movie_id` in `credits`.

In [ ]:
movies_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [ ]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [ ]:
# credits also has a 'title' column; drop it so we don't end up with title_x / title_y
credits = credits.rename(columns={'movie_id': 'id'}).drop(columns='title')

# validate='one_to_one' makes pandas raise an error if any ID were duplicated
movies = movies_raw.merge(credits, on='id', how='inner', validate='one_to_one')
print('After merge:', movies.shape)   # 4803 rows: not one more, not one less

# Keep the release year to tell apart duplicate titles in the web app: "Batman (1989)"
movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year.astype('Int64')

movies = movies[['id', 'title', 'year', 'overview', 'keywords', 'genres', 'cast', 'crew']]
movies.isna().sum()

After merge: (4803, 22)


id          0
title       0
year        1
overview    3
keywords    0
genres      0
cast        0
crew        0
dtype: int64

In [ ]:
# Only 'overview' has missing values (3 movies with no overview): drop them
movies = movies.dropna(subset=['overview'])

# KEY FIX: after dropna the index has gaps (..., 2657, 2659, ...).
# The similarity matrix is indexed by POSITION (row 0, 1, 2...), so the DataFrame
# index must match the position. Without this reset_index, almost half of the
# movies were getting the recommendations of the movie next to them.
movies = movies.reset_index(drop=True)
print(movies.shape)

# Duplicate titles that are now separate movies (with their year):
movies[movies['title'].duplicated(keep=False)][['id', 'title', 'year']]

(4800, 8)


,id,title,year
972,72710,The Host,2013
1359,268,Batman,1989
2876,1255,The Host,2006
3646,39269,Out of the Blue,1980
3692,10844,Out of the Blue,2006
4265,2661,Batman,1966


### Step 2: Extract features and build the `tags` column

In [ ]:
def extract_names(obj, limit=None):
    """Turn TMDB's JSON text into a list of names.
    Used for genres, keywords and cast (with limit=3 for the three lead actors)."""
    names = [item['name'] for item in ast.literal_eval(obj)]
    return names[:limit] if limit else names


def extract_director(obj):
    """Return a list with the director (or an empty list if there is none)."""
    for item in ast.literal_eval(obj):
        if item['job'] == 'Director':
            return [item['name']]
    return []


movies['overview'] = movies['overview'].apply(str.split)
movies['genres'] = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast'] = movies['cast'].apply(extract_names, limit=3)
movies['crew'] = movies['crew'].apply(extract_director)

# Keep a copy of the genres to evaluate the model later
movies['genre_list'] = movies['genres']
movies[['title', 'genres', 'cast', 'crew']].head()

,title,genres,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


**Change:** `tags` previously did not include the genres, even though the comment said it
did. All five fields are now added together.

Spaces are also removed from multi-word names (*Science Fiction* → *ScienceFiction*,
*Sam Worthington* → *SamWorthington*). This way the vectorizer treats each one as a single
term and doesn't confuse *Sam Worthington* with any other *Sam*.

In [ ]:
def collapse(names):
    return [name.replace(' ', '') for name in names]

movies['tags'] = (
    movies['overview']
    + movies['genres'].apply(collapse)     # <- NEW: genres are now included
    + movies['keywords'].apply(collapse)
    + movies['cast'].apply(collapse)
    + movies['crew'].apply(collapse)
)

# Stemming: reduce each word to its root (loves/loved/loving -> love)
ps = PorterStemmer()
movies['tags'] = movies['tags'].apply(lambda words: ' '.join(ps.stem(w) for w in words))

movies.loc[0, 'tags'][:300]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplan'

### Step 3: Vectorize and compute cosine similarity

Two vectorizers can be used:

- **CountVectorizer**: counts how many times each word appears.
- **TF-IDF**: the same, but gives less weight to words that appear in a huge number of movies
  (*life*, *find*, *drama*...) and more weight to words typical of only a few
  (*pirat*, *dinosaur*, *ChristopherNolan*...). It usually gives more specific recommendations.

In [ ]:
def build_similarity(tags, kind=VECTORIZER, max_features=MAX_FEATURES):
    if kind == 'tfidf':
        vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    elif kind == 'count':
        vectorizer = CountVectorizer(max_features=max_features, stop_words='english')
    else:
        raise ValueError("kind must be 'tfidf' or 'count'")
    # The matrix is kept sparse (no .toarray()): it uses far less memory and cosine_similarity accepts it
    vectors = vectorizer.fit_transform(tags)
    return cosine_similarity(vectors)

### Evaluation: which vectorizer works better?

There are no "correct answers" for a content-based recommender, so we use an approximate
metric: for each movie, we look at its 10 recommendations and measure how similar their genres
are (**Jaccard** index: shared genres / total genres). A **random** recommender is included as
a baseline.

Note: genres are also part of the `tags`, so this metric **rewards models that put a lot of
weight on genres**. That is exactly what CountVectorizer does (genres are very frequent words)
and what TF-IDF deliberately avoids. Part 3 compares the vectorizers with an external metric
(ratings from real users), which gives the opposite verdict.

In [ ]:
genre_sets = [set(g) for g in movies['genre_list']]

def genre_jaccard_at_k(sim, k=N_RECS):
    sim = sim.copy()
    np.fill_diagonal(sim, -np.inf)                 # don't recommend the movie itself
    top_k = np.argpartition(-sim, k, axis=1)[:, :k]
    scores = []
    for i, recs in enumerate(top_k):
        if not genre_sets[i]:
            continue
        scores.append(np.mean([
            len(genre_sets[i] & genre_sets[j]) / len(genre_sets[i] | genre_sets[j])
            for j in recs
        ]))
    return np.mean(scores)

rng = np.random.default_rng(RANDOM_STATE)
results = {'random (baseline)': genre_jaccard_at_k(rng.random((len(movies), len(movies))))}
for kind, max_features in [('count', 500), ('count', 5000), ('tfidf', 5000)]:
    sim = build_similarity(movies['tags'], kind, max_features)
    results[f'{kind}, {max_features} terms'] = genre_jaccard_at_k(sim)
    del sim

pd.Series(results, name='Genre Jaccard @10').round(3).to_frame()

,Genre Jaccard @10
random (baseline),0.170
"count, 500 terms",0.516
"count, 5000 terms",0.484
"tfidf, 5000 terms",0.358


In [ ]:
# Final model with the configuration chosen above (VECTORIZER, MAX_FEATURES)
similarity = build_similarity(movies['tags'])
similarity.shape

(4800, 4800)

### Step 4: Test the recommendations

In [ ]:
def recommend(title, n=N_RECS, year=None):
    """Return the n movies most similar to `title`.
    If several movies share that title, the year can be given."""
    matches = movies[movies['title'] == title]
    if year is not None:
        matches = matches[matches['year'] == year]
    if matches.empty:
        raise ValueError(f'Movie "{title}" not found')
    if len(matches) > 1:
        print(f'Warning: {len(matches)} movies are called "{title}"; using the first one. Pass year=...')

    pos = matches.index[0]            # after reset_index, index == position in the matrix
    scores = similarity[pos]
    order = np.argsort(-scores)
    order = order[order != pos][:n]   # remove the movie itself
    return movies.loc[order, ['title', 'year']].assign(similarity=scores[order].round(3))

recommend('Avatar')

,title,year,similarity
2403,Aliens,1986,0.252
3723,Falcon Rising,2014,0.226
582,Battle: Los Angeles,2011,0.194
1213,Aliens vs Predator: Requiem,2007,0.189
3603,Apollo 18,2011,0.177
47,Star Trek Into Darkness,2013,0.173
778,Meet Dave,2008,0.166
1201,Predators,2010,0.163
539,Titan A.E.,2000,0.157
942,The Book of Life,2014,0.155


In [ ]:
recommend('The Dark Knight Rises')

,title,year,similarity
65,The Dark Knight,2008,0.461
428,Batman Returns,1992,0.408
299,Batman Forever,1995,0.334
119,Batman Begins,2005,0.324
1359,Batman,1989,0.291
210,Batman & Robin,1997,0.265
3853,"Batman: The Dark Knight Returns, Part 2",2013,0.242
9,Batman v Superman: Dawn of Justice,2016,0.231
2507,Slow Burn,2005,0.202
1181,JFK,1991,0.132


In [ ]:
recommend('Batman', year=1989)

,title,year,similarity
210,Batman & Robin,1997,0.425
428,Batman Returns,1992,0.324
3,The Dark Knight Rises,2012,0.291
119,Batman Begins,2005,0.269
299,Batman Forever,1995,0.225
65,The Dark Knight,2008,0.208
9,Batman v Superman: Dawn of Justice,2016,0.200
813,Superman,1978,0.149
30,Spider-Man 2,2004,0.134
1469,Chill Factor,1999,0.131


### Step 5: Save the model

- `model.pkl`: only the columns the web app needs (`id`, `title`, `year`).
- `similarity.pkl`: the matrix in **float32**. It takes half the space of float64 (~92 MB instead
  of ~185 MB), which is below GitHub's 100 MB per-file limit, and the lost precision does not
  change the order of the recommendations in practice.

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(movies[['id', 'title', 'year']], f)

with open('similarity.pkl', 'wb') as f:
    pickle.dump(similarity.astype(np.float32), f)

for name in ['model.pkl', 'similarity.pkl']:
    print(f'{name}: {os.path.getsize(name) / 1e6:.1f} MB')

model.pkl: 0.2 MB
similarity.pkl: 92.2 MB


---
## Part 2 (bonus) · User-based collaborative filtering

Collaborative filtering ignores the content of the movies and only looks at **who rated what**.
The idea: to predict how much Anna will like a movie, we find the users whose ratings are
similar to Anna's (her "neighbours") and average what they gave it.

TMDB 5000 has no individual ratings, so we use **MovieLens ml-latest-small**
(100,836 ratings from 610 users). Its `links.csv` file includes the TMDB ID of each movie,
which lets us combine it with the content model in the hybrid part.

In [ ]:
ML_DIR = os.path.join(DATA_DIR, 'ml-latest-small')
if not os.path.exists(ML_DIR):
    zip_path = ML_DIR + '.zip'
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip', zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

ratings = pd.read_csv(os.path.join(ML_DIR, 'ratings.csv'))
links = pd.read_csv(os.path.join(ML_DIR, 'links.csv'))
ml_movies = pd.read_csv(os.path.join(ML_DIR, 'movies.csv'))
print(ratings.shape, 'users:', ratings['userId'].nunique(), 'movies:', ratings['movieId'].nunique())
ratings.head()

(100836, 4) users: 610 movies: 9724


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### Train/test split

For each user we hold out 20% of their ratings as the **test** set. The model only sees the
remaining 80%, and then we check whether it gets the unseen ratings right.

In [ ]:
test = ratings.groupby('userId').sample(frac=0.2, random_state=RANDOM_STATE)
train = ratings.drop(test.index)
print('train:', len(train), ' test:', len(test))

# User × movie matrix (0 = not rated)
user_ids = np.sort(ratings['userId'].unique())
item_ids = np.sort(ratings['movieId'].unique())
user_pos = pd.Series(np.arange(len(user_ids)), index=user_ids)
item_pos = pd.Series(np.arange(len(item_ids)), index=item_ids)

R = np.zeros((len(user_ids), len(item_ids)))
R[user_pos[train['userId']].to_numpy(), item_pos[train['movieId']].to_numpy()] = train['rating']
rated = R > 0
print('Matrix:', R.shape, f'- {rated.mean():.1%} filled')

train: 80672  test: 20164
Matrix: (610, 9724) - 1.4% filled


### User-user similarity

Every user rates on their own scale: for one user a 3 is "fine", for another it means "bad".
That's why we subtract each user's mean before comparing (*centering*). Cosine similarity on
centered ratings is practically the **Pearson correlation**.

In [ ]:
user_mean = R.sum(axis=1) / rated.sum(axis=1)
R_centered = np.where(rated, R - user_mean[:, None], 0.0)

user_sim = cosine_similarity(R_centered)
np.fill_diagonal(user_sim, 0)   # a user is not their own neighbour

### Prediction

For user *u* we take their *k* most similar neighbours (with positive similarity) and predict:

$$\hat r_{u,i} = \bar r_u + \frac{\sum_{v} \text{sim}(u,v)\,(r_{v,i} - \bar r_v)}{\sum_{v} \text{sim}(u,v) + \lambda}$$

where the sums run over the neighbours who rated movie *i*. The λ term (*shrinkage*) prevents
a movie rated by a single neighbour from getting an extreme prediction.

In [ ]:
def predict_user_cf(u, k=30, shrinkage=1.0):
    """Predict user u's rating (by position) for ALL movies."""
    sims = user_sim[u]
    neighbours = np.argpartition(-sims, k)[:k]
    neighbours = neighbours[sims[neighbours] > 0]
    w = sims[neighbours]
    numerator = w @ R_centered[neighbours]
    denominator = w @ rated[neighbours]      # sum of similarities of those who rated it
    pred = user_mean[u] + numerator / (denominator + shrinkage)
    return np.clip(pred, 0.5, 5.0)

def predict_all(k, shrinkage=1.0):
    return np.vstack([predict_user_cf(u, k, shrinkage) for u in range(len(user_ids))])

### Evaluation: RMSE and MAE

We compare against two very simple baselines: always predicting the global mean, and always
predicting the user's mean. A good collaborative model has to beat both.

In [ ]:
t_u = user_pos[test['userId']].to_numpy()
t_i = item_pos[test['movieId']].to_numpy()
y = test['rating'].to_numpy()

def rmse(pred): return np.sqrt(np.mean((pred - y) ** 2))
def mae(pred): return np.mean(np.abs(pred - y))

global_mean = train['rating'].mean()
rows = {
    'global mean': (rmse(np.full_like(y, global_mean)), mae(np.full_like(y, global_mean))),
    'user mean': (rmse(user_mean[t_u]), mae(user_mean[t_u])),
}
for k in [10, 30, 60, 100]:
    P = predict_all(k)
    rows[f'user-based CF, k={k}'] = (rmse(P[t_u, t_i]), mae(P[t_u, t_i]))

pd.DataFrame(rows, index=['RMSE', 'MAE']).T.round(4)

,RMSE,MAE
global mean,1.0498,0.8307
user mean,0.9518,0.7404
"user-based CF, k=10",0.9147,0.7039
"user-based CF, k=30",0.8973,0.6865
"user-based CF, k=60",0.8901,0.6793
"user-based CF, k=100",0.8868,0.6758


In [ ]:
K_BEST = 60
P_cf = predict_all(K_BEST)

> **Alternative with the `surprise` library.** The brief mentions it (SVD, KNNBasic).
> It is not in `requirements.txt` because it often fails to install with NumPy 2 and Python 3.12.
> If you want to try it, use a separate environment with `numpy<2` and `pip install scikit-surprise`:
>
> ```python
> from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
> from surprise.model_selection import train_test_split
> data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], Reader(rating_scale=(0.5, 5)))
> trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
> for algo in [KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': True}), SVD(random_state=42)]:
>     algo.fit(trainset)
>     accuracy.rmse(algo.test(testset))
> ```

---
## Part 3 (bonus) · Hybrid recommender

We combine the two models for each user:

- **Collaborative score**: the rating predicted above.
- **Content score**: how similar each movie is (according to the TMDB similarity matrix) to
  the movies the user rated 4 or higher in the training set.

Both are scaled to [0, 1] for each user and blended with a weight α:

$$\text{hybrid} = \alpha \cdot \text{collaborative} + (1-\alpha) \cdot \text{content}$$

α = 1 is pure collaborative filtering and α = 0 is pure content. Only movies present in both
datasets are used (they are linked through the TMDB ID in `links.csv`).

In [ ]:
# Link each MovieLens movie to its row in the TMDB similarity matrix
tmdb_row = pd.Series(movies.index, index=movies['id'])
links = links.dropna(subset=['tmdbId']).astype({'tmdbId': int})
links['tmdb_row'] = links['tmdbId'].map(tmdb_row)
item_tmdb_row = links.set_index('movieId')['tmdb_row'].reindex(item_ids).to_numpy()

shared = np.where(~np.isnan(item_tmdb_row))[0]            # columns of R that exist in TMDB
shared_tmdb = item_tmdb_row[shared].astype(int)
content_sim = similarity[np.ix_(shared_tmdb, shared_tmdb)]  # similarity between those movies
print('Movies in both datasets:', len(shared))

Movies in both datasets: 3536


In [ ]:
def minmax(x):
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 0 else np.zeros_like(x)

def user_scores(u, csim=None):
    """Collaborative and content scores (scaled) over the shared movies.
    csim lets us try a different content matrix; content_sim is used by default."""
    csim = content_sim if csim is None else csim
    cf = minmax(P_cf[u, shared])
    liked = np.where(R[u, shared] >= 4)[0]             # what the user liked in training
    content = csim[:, liked].mean(axis=1) if len(liked) else np.zeros(len(shared))
    return cf, minmax(content)

def hybrid_scores(u, alpha, csim=None):
    cf, content = user_scores(u, csim)
    return alpha * cf + (1 - alpha) * content

### Evaluation: precision@10 and recall@10

Now we measure whether the system **recommends** well, not whether it predicts the exact
rating. For each user we recommend 10 movies they haven't seen in training and count how many
of them are among the movies they rated 4 or higher in the test set.

- **precision@10**: of the 10 recommended movies, what fraction they liked.
- **recall@10**: of the movies they liked, what fraction appears in the 10.

A **popularity** baseline (recommend the most-rated movies) is included; on MovieLens it is
often surprisingly hard to beat.

In [ ]:
test_liked = test[test['rating'] >= 4]
relevant = {
    user_pos[u]: set(item_pos[g['movieId']].to_numpy())
    for u, g in test_liked.groupby('userId')
}
shared_set_pos = {c: n for n, c in enumerate(shared)}
popularity = minmax(rated[:, shared].sum(axis=0).astype(float))

def evaluate(score_fn, k=10):
    precisions, recalls = [], []
    for u, rel_items in relevant.items():
        rel = {shared_set_pos[i] for i in rel_items if i in shared_set_pos}
        if not rel:
            continue
        scores = score_fn(u).astype(float).copy()
        scores[rated[u, shared]] = -np.inf           # don't recommend what they've already seen
        top = np.argpartition(-scores, k)[:k]
        hits = len(rel.intersection(top))
        precisions.append(hits / k)
        recalls.append(hits / len(rel))
    return np.mean(precisions), np.mean(recalls)

rows = {'popularity (baseline)': evaluate(lambda u: popularity)}
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    label = {0.0: 'pure content', 1.0: 'pure collaborative'}.get(alpha, 'hybrid')
    rows[f'α = {alpha} ({label})'] = evaluate(lambda u, a=alpha: hybrid_scores(u, a))

pd.DataFrame(rows, index=['precision@10', 'recall@10']).T.round(4)

,precision@10,recall@10
popularity (baseline),0.1143,0.1156
α = 0.0 (pure content),0.0236,0.0365
α = 0.25 (hybrid),0.0382,0.0592
α = 0.5 (hybrid),0.0971,0.1281
α = 0.75 (hybrid),0.1426,0.1568
α = 1.0 (pure collaborative),0.1267,0.1353


### Back to the question: TF-IDF or CountVectorizer?

With MovieLens we finally have an **external** way to judge the content model: whether its
similarities help predict which movies real users like. We repeat the evaluation with each
vectorizer (pure content, and the hybrid with α = 0.75).

In [ ]:
rows = {}
for kind, max_features in [('count', 500), ('count', 5000), ('tfidf', 5000)]:
    sim = build_similarity(movies['tags'], kind, max_features)
    csim = sim[np.ix_(shared_tmdb, shared_tmdb)]
    rows[f'{kind}, {max_features} terms'] = [
        evaluate(lambda u: hybrid_scores(u, 0.0, csim))[0],
        evaluate(lambda u: hybrid_scores(u, 0.75, csim))[0],
    ]
    del sim, csim

pd.DataFrame(rows, index=['precision@10 pure content', 'precision@10 hybrid α=0.75']).T.round(4)

,precision@10 pure content,precision@10 hybrid α=0.75
"count, 500 terms",0.0100,0.1192
"count, 5000 terms",0.0093,0.1207
"tfidf, 5000 terms",0.0236,0.1426


### Example: hybrid recommendations for one user

In [ ]:
ALPHA = 0.5
ml_titles = ml_movies.set_index('movieId')['title']

def recommend_for_user(user_id, n=N_RECS, alpha=ALPHA):
    u = user_pos[user_id]
    scores = hybrid_scores(u, alpha)
    scores[rated[u, shared]] = -np.inf
    top = np.argsort(-scores)[:n]
    return pd.DataFrame({
        'title': ml_titles.loc[item_ids[shared[top]]].to_numpy(),
        'hybrid score': scores[top].round(3),
    })

u = user_pos[1]
favourites = np.argsort(-R[u])[:5]
print("User 1's favourites:", list(ml_titles.loc[item_ids[favourites]]))
recommend_for_user(1)

User 1's favourites: ['Seven (a.k.a. Se7en) (1995)', 'Usual Suspects, The (1995)', 'X-Men (2000)', 'M*A*S*H (a.k.a. MASH) (1970)', 'Blazing Saddles (1974)']


,title,hybrid score
0,"Fugitive, The (1993)",0.765
1,"Shawshank Redemption, The (1994)",0.749
2,Pulp Fiction (1994),0.748
3,Micmacs (Micmacs à tire-larigot) (2009),0.734
4,Brazil (1985),0.732
5,Star Wars: Episode IV - A New Hope (1977),0.731
6,"Great Escape, The (1963)",0.729
7,"Godfather, The (1972)",0.729
8,Star Wars: Episode VI - Return of the Jedi (1983),0.726
9,Dr. Strangelove or: How I Learned to Stop Worr...,0.719


### Conclusions

- **Content.** Merging on ID, adding the genres and resetting the index fix the bugs in the
  original version. Between vectorizers, the genre metric prefers CountVectorizer, but the
  real-user metric prefers **TF-IDF** (more than twice the precision for pure content),
  so it is the default. Lesson: a poorly chosen metric can lead to the wrong decision.
- **Collaborative.** User-based filtering beats the baselines in RMSE
  (≈0.89 vs 0.95 for the user mean) and beats popularity in precision@10.
- **Pure content** recommends worse than popularity: it finds *similar* movies, but it
  doesn't know which ones are good or which ones people actually watch.
- **Hybrid.** With α = 0.75 it gets the best precision@10 and recall@10 of all: the
  collaborative part contributes "what people like you enjoy" and the content part tunes it
  towards the user's taste. With a low α, content dominates and results get worse.

Note: the numbers depend on the random split (`RANDOM_STATE`); with a different seed they
change slightly, but the ranking between models should hold.